# Lipi: run experiments on a hosted GPU

Works on Google Colab, Kaggle, Lightning AI Studios and Azure ML notebooks. Switch the runtime to a GPU, fill in the first cell, then run all cells. If the session disconnects, run all cells again: finished jobs are skipped.

In [ ]:
REPO_URL = "https://github.com/<user>/lipi.git"      # the code
DATA_URL = ""          # direct link or Google Drive link to the zip with canonical/ inside
                       # (leave empty if the images are attached another way, see below)
WHO      = "your-name"
WHAT     = "--group loso"                           # e.g. "--group resolution --shard 1/2" or "--jobs ancestry-conv6-56px-s0"
KEEP_RUNS_IN = ""      # folder that survives a disconnect, e.g. "/content/drive/MyDrive/lipi_runs" on Colab;
                       # leave empty on Kaggle / Lightning / Azure, where the working folder is kept

In [ ]:
import os, subprocess, sys
if not os.path.exists("lipi"):
    subprocess.run(["git", "clone", REPO_URL, "lipi"], check=True)
os.chdir("lipi")
subprocess.run(["git", "pull"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to a GPU")

### Images
Either set `DATA_URL` above, or attach the images yourself:
- **Kaggle:** add the dataset to the notebook, then `!ln -s /kaggle/input/<dataset-name>/canonical data/canonical`
- **Any platform:** upload the zip next to this notebook and `!unzip -q <file>.zip -d data/`

In [ ]:
if DATA_URL and not os.path.exists("data/canonical"):
    if "drive.google.com" in DATA_URL:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        subprocess.run([sys.executable, "-m", "gdown", "--fuzzy", DATA_URL, "-O", "lipi_data.zip"], check=True)
    else:
        subprocess.run(["wget", "-q", "-O", "lipi_data.zip", DATA_URL], check=True)
    subprocess.run(["unzip", "-q", "lipi_data.zip", "-d", "data/"], check=True)
assert os.path.exists("data/canonical/brahmi"), "images not found: data/canonical/<script>/<character>/*.png is expected"

In [ ]:
if KEEP_RUNS_IN:
    if KEEP_RUNS_IN.startswith("/content/drive"):
        from google.colab import drive; drive.mount("/content/drive")
    os.makedirs(KEEP_RUNS_IN, exist_ok=True)
    os.environ["LIPI_RUNS_DIR"] = KEEP_RUNS_IN
!python run.py smoke

In [ ]:
!python run.py train {WHAT} --who {WHO}

In [ ]:
!python run.py status
!python run.py pack --who {WHO}
# download the lipi_runs_*.zip file from the lipi folder and send it back